# 🧠 Exploration & visualisation des volumes IRM

Notebook d'exploration du dataset **ehl-paris-medical-image-retrieval**.
Objectif : **comprendre la structure 3D** des volumes `.nii` via plusieurs angles de vue.

| Section | Visualisation |
|---|---|
| 1 | Setup & découverte des données |
| 2 | Métadonnées (formes, intensités) par dataset |
| 3 | Vue tri-planaire (axial / coronal / sagittal) |
| 4 | Montage tranche par tranche (grille) |
| 5 | Bande de tranches colorée par profondeur |
| 6 | Curseur interactif (Plotly) |
| 7 | Projections d'intensité maximale (MIP) |
| 8 | **3D : tranches empilées différenciées par profondeur** |
| 9 | **3D : isosurface (marching cubes)** |
| 10 | **3D : rendu volumique** |
| 11 | Comparaison paire query ↔ target |


## 1. Setup & découverte des données

In [ ]:
import os, glob, math
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib import cm
import plotly.graph_objects as go

plt.rcParams['figure.dpi'] = 110
plt.rcParams['image.cmap'] = 'gray'

# Racine des données (relative au dossier notebooks/)
DATA_ROOT = os.path.join('..', 'data', 'ehl-paris-medical-image-retrieval')
assert os.path.isdir(DATA_ROOT), f'Introuvable: {os.path.abspath(DATA_ROOT)}'

vol_paths = sorted(glob.glob(os.path.join(DATA_ROOT, 'dataset*', 'images', '**', '*.nii*'), recursive=True))
print(f'{len(vol_paths)} volumes trouvés')

def rel(p):
    return os.path.relpath(p, DATA_ROOT).replace('\\', '/')

# Index : (dataset, split, role) -> liste de chemins
from collections import defaultdict
index = defaultdict(list)
for p in vol_paths:
    parts = rel(p).split('/')           # datasetX/images/split/role/file
    index[(parts[0], parts[2], parts[3])].append(p)

print('\nRépartition :')
for k in sorted(index):
    print(f'  {k[0]:8s} {k[1]:5s} {k[2]:8s} : {len(index[k]):4d}')


### Helpers de chargement & normalisation

In [ ]:
def load_volume(path):
    """Charge un .nii -> np.float32 (X, Y, Z)."""
    return np.asarray(nib.load(path).get_fdata(), dtype=np.float32)

def norm01(v, p_lo=0.5, p_hi=99.5):
    """Normalise en [0,1] avec clipping par percentiles (robuste aux outliers)."""
    lo, hi = np.percentile(v[v > 0], [p_lo, p_hi]) if np.any(v > 0) else (v.min(), v.max())
    if hi <= lo:
        hi = lo + 1e-6
    return np.clip((v - lo) / (hi - lo), 0, 1)

# Volume d'exemple : 1ère query de dataset1/train (sinon 1er volume dispo)
def pick(ds='dataset1', split='train', role='queries'):
    for key in [(ds, split, role)] + list(index):
        if index[key]:
            return index[key][0]
    return vol_paths[0]

EXAMPLE = pick()
print('Exemple :', rel(EXAMPLE))
vol = norm01(load_volume(EXAMPLE))
print('shape', vol.shape, '| min/max', vol.min(), vol.max())


## 2. Métadonnées par dataset
Formes et intensités diffèrent selon le dataset — important pour le preprocessing.

In [ ]:
import nibabel as nib
rows = []
for k in sorted(index):
    p = index[k][0]
    img = nib.load(p)
    d = img.get_fdata()
    rows.append((f'{k[0]}/{k[1]}/{k[2]}', d.shape, tuple(round(z,2) for z in img.header.get_zooms()),
                 round(float(d.min()),1), round(float(d.max()),1)))

print(f"{'groupe':28s} {'shape':22s} {'zooms':16s} {'min':>7s} {'max':>8s}")
print('-'*84)
for name, shape, zooms, mn, mx in rows:
    print(f'{name:28s} {str(shape):22s} {str(zooms):16s} {mn:7.1f} {mx:8.1f}')


In [ ]:
# Histogramme des intensités (voxels > 0) pour un volume de chaque dataset
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, ds in zip(axes, ['dataset1', 'dataset2', 'dataset3']):
    p = pick(ds, 'val', 'queries')
    v = load_volume(p)
    ax.hist(v[v > 0].ravel(), bins=120, color='steelblue')
    ax.set_title(f'{ds} — intensités (>0)')
    ax.set_yscale('log')
plt.tight_layout(); plt.show()


## 3. Vue tri-planaire (coupes centrales)
Les 3 plans orthogonaux passant par le centre du volume.

In [ ]:
def triplanar(v, title=''):
    cx, cy, cz = [s // 2 for s in v.shape]
    fig, ax = plt.subplots(1, 3, figsize=(14, 5))
    ax[0].imshow(np.rot90(v[cx, :, :])); ax[0].set_title(f'Sagittal (x={cx})')
    ax[1].imshow(np.rot90(v[:, cy, :])); ax[1].set_title(f'Coronal (y={cy})')
    ax[2].imshow(np.rot90(v[:, :, cz])); ax[2].set_title(f'Axial (z={cz})')
    for a in ax: a.axis('off')
    fig.suptitle(title, fontsize=13); plt.tight_layout(); plt.show()

triplanar(vol, rel(EXAMPLE))


## 4. Montage tranche par tranche
Grille de coupes axiales régulièrement espacées le long de l'axe Z.

In [ ]:
def montage(v, axis=2, n=20, title=''):
    nslices = v.shape[axis]
    idxs = np.linspace(nslices*0.1, nslices*0.9, n).astype(int)
    cols = 5; rows = math.ceil(n/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.4, rows*2.4))
    for ax, i in zip(axes.ravel(), idxs):
        sl = np.take(v, i, axis=axis)
        ax.imshow(np.rot90(sl)); ax.set_title(f'{i}', fontsize=8); ax.axis('off')
    for ax in axes.ravel()[len(idxs):]: ax.axis('off')
    fig.suptitle(title or f'Montage axe={axis}', fontsize=13)
    plt.tight_layout(); plt.show()

montage(vol, axis=2, n=20, title=f'Coupes axiales — {rel(EXAMPLE)}')


In [ ]:
# Les 3 axes côte à côte (une rangée par axe)
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
names = ['Sagittal (x)', 'Coronal (y)', 'Axial (z)']
for r, axis in enumerate([0, 1, 2]):
    n = vol.shape[axis]
    for c, i in enumerate(np.linspace(n*0.15, n*0.85, 8).astype(int)):
        sl = np.take(vol, i, axis=axis)
        axes[r, c].imshow(np.rot90(sl)); axes[r, c].axis('off')
        if c == 0: axes[r, c].set_ylabel(names[r], fontsize=11)
    axes[r, 0].set_title(names[r], fontsize=11, loc='left')
plt.tight_layout(); plt.show()


## 5. Bande de tranches colorée par profondeur
Chaque coupe est teintée selon sa position Z (bleu = avant, rouge = arrière) :
on visualise immédiatement **où** se situe chaque tranche dans le volume.

In [ ]:
n = 12
zs = np.linspace(vol.shape[2]*0.15, vol.shape[2]*0.85, n).astype(int)
colors = cm.plasma(np.linspace(0, 1, n))
fig, axes = plt.subplots(1, n, figsize=(n*1.5, 2.4))
for ax, z, col in zip(axes, zs, colors):
    ax.imshow(np.rot90(vol[:, :, z]))
    for spine in ax.spines.values():
        spine.set_edgecolor(col); spine.set_linewidth(4)
    ax.set_title(f'z={z}', color=col, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Coupes axiales colorées par profondeur (plasma)', fontsize=12)
plt.tight_layout(); plt.show()


## 6. Curseur interactif (Plotly)
Fais glisser le curseur pour parcourir les coupes axiales.

In [ ]:
step = max(1, vol.shape[2] // 40)
zs = list(range(0, vol.shape[2], step))
frames = [go.Frame(data=[go.Heatmap(z=np.rot90(vol[:, :, z]), colorscale='gray', showscale=False)],
                   name=str(z)) for z in zs]
fig = go.Figure(data=frames[0].data, frames=frames)
fig.update_layout(
    title=f'Coupes axiales — {rel(EXAMPLE)}',
    width=520, height=560, yaxis=dict(scaleanchor='x'),
    sliders=[dict(active=0, steps=[dict(method='animate',
        args=[[f.name], dict(mode='immediate', frame=dict(duration=0, redraw=True))],
        label=f.name) for f in frames])],
    updatemenus=[dict(type='buttons', showactive=False, x=0.1, y=1.15,
        buttons=[dict(label='▶ Play', method='animate',
            args=[None, dict(frame=dict(duration=80, redraw=True), fromcurrent=True)])])])
fig.show()


## 7. Projections d'intensité maximale (MIP)
Projection du max le long de chaque axe — vue d'ensemble de l'anatomie.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(np.rot90(vol.max(axis=0)), cmap='magma'); ax[0].set_title('MIP sagittale (max sur X)')
ax[1].imshow(np.rot90(vol.max(axis=1)), cmap='magma'); ax[1].set_title('MIP coronale (max sur Y)')
ax[2].imshow(np.rot90(vol.max(axis=2)), cmap='magma'); ax[2].set_title('MIP axiale (max sur Z)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()


## 8. 🧊 3D — tranches empilées différenciées par profondeur
Quelques coupes axiales placées à leur **vraie hauteur Z** dans l'espace 3D, chacune
sur un plan coloré : on perçoit l'empilement et la séparation entre tranches.

In [ ]:
n = 8
zs = np.linspace(vol.shape[2]*0.2, vol.shape[2]*0.8, n).astype(int)
ds_factor = 3  # sous-échantillonnage pour la fluidité
X, Y = np.mgrid[0:vol.shape[0]:ds_factor, 0:vol.shape[1]:ds_factor]
fig = go.Figure()
for k, z in enumerate(zs):
    sl = vol[::ds_factor, ::ds_factor, z]
    fig.add_surface(
        x=X, y=Y, z=np.full_like(sl, z, dtype=float),
        surfacecolor=sl, colorscale='gray', showscale=False,
        opacity=0.92, name=f'z={z}',
        contours=dict(z=dict(show=True, color=f'rgba({int(255*k/n)},80,{int(255*(1-k/n))},0.9)', width=3)))
fig.update_layout(title='Coupes axiales empilées en 3D (bordure colorée = profondeur)',
                  width=700, height=650,
                  scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z (profondeur)',
                             aspectmode='data'))
fig.show()


## 9. 🧊 3D — isosurface (marching cubes)
Surface 3D du cerveau extraite par seuillage (algorithme *marching cubes*).

In [ ]:
from skimage import measure

vol_ds = vol[::2, ::2, ::2]          # sous-échantillonnage pour la vitesse
level = 0.25
verts, faces, _, _ = measure.marching_cubes(vol_ds, level=level)
fig = go.Figure(data=[go.Mesh3d(
    x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    intensity=verts[:, 2], colorscale='Viridis',
    opacity=1.0, showscale=True, colorbar=dict(title='Z'))])
fig.update_layout(title=f'Isosurface (seuil={level})', width=700, height=650,
                  scene=dict(aspectmode='data'))
fig.show()


## 10. 🧊 3D — rendu volumique
Rendu volumétrique semi-transparent : la couleur encode l'intensité, l'opacité révèle l'intérieur.

In [ ]:
vol_ds = vol[::3, ::3, ::3]
Xv, Yv, Zv = np.mgrid[0:vol_ds.shape[0], 0:vol_ds.shape[1], 0:vol_ds.shape[2]]
fig = go.Figure(data=go.Volume(
    x=Xv.ravel(), y=Yv.ravel(), z=Zv.ravel(), value=vol_ds.ravel(),
    isomin=0.15, isomax=1.0, opacity=0.08, surface_count=18,
    colorscale='Hot', caps=dict(x_show=False, y_show=False, z_show=False)))
fig.update_layout(title='Rendu volumique', width=700, height=650, scene=dict(aspectmode='data'))
fig.show()


## 11. Comparaison paire query ↔ target
Le dataset1 fournit `train_pairs.csv` : une query (T1 post-contrast) appariée à une target (T2).
On compare les coupes centrales des deux modalités.

In [ ]:
import csv
pairs_csv = os.path.join(DATA_ROOT, 'dataset1', 'train_pairs.csv')
if os.path.exists(pairs_csv):
    with open(pairs_csv) as f:
        row = next(csv.DictReader(f))
    qp = os.path.join(DATA_ROOT, *row['query_image'].split('/'))
    tp = os.path.join(DATA_ROOT, *row['target_image'].split('/'))
    # les chemins du csv peuvent finir en .nii.gz alors que les fichiers sont .nii
    for cand in (qp, qp.replace('.nii.gz', '.nii')):
        if os.path.exists(cand): qp = cand; break
    for cand in (tp, tp.replace('.nii.gz', '.nii')):
        if os.path.exists(cand): tp = cand; break

    q = norm01(load_volume(qp)); t = norm01(load_volume(tp))
    fig, ax = plt.subplots(2, 3, figsize=(13, 8))
    for r, (v, lab, mod) in enumerate([(q, 'QUERY', row['query_modality']),
                                       (t, 'TARGET', row['target_modality'])]):
        cx, cy, cz = [s//2 for s in v.shape]
        ax[r,0].imshow(np.rot90(v[cx,:,:])); ax[r,0].set_title(f'{lab} sagittal')
        ax[r,1].imshow(np.rot90(v[:,cy,:])); ax[r,1].set_title(f'{lab} coronal — {mod}')
        ax[r,2].imshow(np.rot90(v[:,:,cz])); ax[r,2].set_title(f'{lab} axial')
        for a in ax[r]: a.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('train_pairs.csv introuvable')


---
✅ **Fin du tour.** Astuce : change `EXAMPLE` (cellule §1) — par ex.
`EXAMPLE = pick('dataset3', 'val', 'queries')` — puis ré-exécute pour explorer un autre volume.